# Local Spark Connect

An interactive Spark session from the laptop against any Spark Connect endpoint. The client is only `pyspark-client`:
no JVM, no full PySpark.

    pip install pyspark-client ipykernel
    pip install pysail azure-identity        # only for the local Sail fallback, plus `az login`

Set `REMOTE` below (or `SPARK_REMOTE` in the repo-root `.env`) to a Spark Connect URL such as `sc://host:15002/;token=...`.
Leave both empty to start a local Sail server with the OneLake Iceberg catalog, the same setup as
`platforms/lakesail/run.py` (needs `ONELAKE_WAREHOUSE=<workspace id>/<lakehouse id>` in `.env`).

In [ ]:
REMOTE = ""   # sc://host:port/;token=...   empty: SPARK_REMOTE from .env, else a local Sail server

In [ ]:
import os
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists())
for line in open(ROOT / ".env"):
    if "=" in line and not line.lstrip().startswith("#"):
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

server = None
remote = REMOTE or os.environ.get("SPARK_REMOTE")
if not remote:
    # Local Sail, as in platforms/lakesail/run.py. One Entra token for https://storage.azure.com/ covers both the
    # catalog and the files: bearer_token for the catalog, AZURE_STORAGE_TOKEN for storage (no credential vending).
    from azure.identity import AzureCliCredential
    token = AzureCliCredential().get_token("https://storage.azure.com/.default").token
    # Read once at server start: must be set before SparkConnectServer() exists.
    os.environ["SAIL_CATALOG__LIST"] = (
        f'[{{type="onelake", name="onelake", url="{os.environ["ONELAKE_WAREHOUSE"]}", '
        f'api="iceberg", bearer_token="{token}", '
        f'table_cache_type="session", table_cache_ttl_secs=300, '
        f'database_cache_type="session", database_cache_ttl_secs=300}}]'
    )
    os.environ["SAIL_CATALOG__DEFAULT_CATALOG"] = "onelake"
    os.environ["AZURE_STORAGE_TOKEN"] = token
    os.environ.setdefault("RUST_LOG", "error")
    from pysail.spark import SparkConnectServer
    server = SparkConnectServer()
    server.start()
    remote = f"sc://localhost:{server.listening_address[1]}"

from pyspark.sql import SparkSession
spark = SparkSession.builder.remote(remote).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.ansi.enabled", "false")   # Spark 3.5 semantics, as the jobs expect
print("connected to", remote.split(";")[0], "| client", spark.version)

In [ ]:
spark.sql("SELECT version()").show(truncate=False)
spark.sql("SHOW NAMESPACES").show()
spark.range(5).selectExpr("id", "id * id AS sq").show()

In [ ]:
spark.stop()
if server:
    server.stop()